In [13]:
import pandas as pd

## THIS SECTION IS TAKING REGISTRATION DATA FROM TWO .PDF FILES

This is third attempt to get the .pdf imported correctly.  

Step 1: Import libraries with tabula first

In [ ]:
import tabula
import pandas as pd

Step 2: Read the .pdf

In [15]:
# cell 2: read tables from PDF (returns a list of DataFrames)
pdf_path = "C:/Users/mutch/Democratic_Voter_Shift/data/raw/2020_voterreg.pdf"                    
dfs = tabula.read_pdf(pdf_path, 
    pages="all",
    guess=False,
    stream=True,
    multiple_tables=False
)

Step 3: Clean the table

In [24]:
df = dfs[0].copy()

# remove the first three rows
df = df.iloc[3:].reset_index(drop=True)

# remove the 4th column by position
df = df.drop(df.columns[3], axis=1)

# rename the columns based on what you described
df.columns = ["County_Dem_Rep", "Other Parties", "All Parties"]

# split the combined first column into 3 fields
split_cols = df["County_Dem_Rep"].str.split(r"\s+", n=2, expand=True)
split_cols.columns = ["County", "Democrat", "Republican"]

# Add data validation and cleaning before conversion
# Check for non-numeric values and handle them
def safe_convert_to_int(series):
    # Remove commas and convert to string first
    cleaned = series.str.replace(",", "", regex=False)
    # Use pd.to_numeric with errors='coerce' to handle invalid values
    return pd.to_numeric(cleaned, errors='coerce').fillna(0).astype(int)

# Apply safe conversion to numeric columns
split_cols["Democrat"] = safe_convert_to_int(split_cols["Democrat"])
split_cols["Republican"] = safe_convert_to_int(split_cols["Republican"])

df["Other Parties"] = safe_convert_to_int(df["Other Parties"])
df["All Parties"] = safe_convert_to_int(df["All Parties"])

# final cleaned table
cleaned = pd.concat([split_cols, df[["Other Parties", "All Parties"]]], axis=1)

In [25]:
# cell 4: save the first table
if dfs:
    cleaned.to_csv("C:/Users/mutch/Democratic_Voter_Shift/data/raw/VoterReg2020_cleaned.csv", index=False)

NOTE: THIS WORKED, but the .csv must have the first row and the last row (Totals) removed.  I also have to make a column for 'year' and make every entry 2020

In [26]:
# load file
file_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/VoterReg2020_cleaned.csv"
df = pd.read_csv(file_path)

# remove rows 1 and 69
# assuming you mean row numbers as shown in the file, starting at 1
df = df.drop(index=[0, 68]).reset_index(drop=True)

# normalize column names: lowercase + underscores
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(' ', '_', regex=False)
)

# insert 'year' column after 'county'
if 'county' in df.columns:
    county_pos = df.columns.get_loc('county')
    df.insert(county_pos + 1, 'year', 2020)
else:
    raise KeyError("Expected a 'county' column, but it was not found.")

# save cleaned file
output_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/VoterReg2020_cleaned_v2.csv"
df.to_csv(output_path, index=False)

df.head()

,county,year,democrat,republican,other_parties,all_parties
0,Adams,2020,20063,40781,11544,72388
1,Allegheny,2020,539595,267832,134829,942256
2,Armstrong,2020,13659,25979,5188,44826
3,Beaver,2020,54281,47214,15456,116951
4,Bedford,2020,7264,23729,3246,34239


### SUMMARY - VoterReg2020_cleaned_v2.csv is cleaned and ready for use.